In [ ]:
# Un engine separa el análisis de la tecnología concreta usada para conectarse a los datos.

from pathlib import Path
import json

import pandas as pd
from sqlalchemy import create_engine, text

ACTIVITY_DIR = next(candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / "data").is_dir() and (candidate / "submission").is_dir())
DATABASE_FILE = ACTIVITY_DIR / "data" / "scopus_proptech.db"
SUBMISSION_DIR = ACTIVITY_DIR / "submission"

In [ ]:
# SQLite hoy ──► SQLAlchemy engine ──► otra base compatible mañana

engine = create_engine(f"sqlite:///{DATABASE_FILE}")

In [ ]:
# El parámetro mantiene separada la pregunta analítica de su valor de ejecución.

recent_sources = pd.read_sql_query(
    text(
        """
        SELECT s.source_title, COUNT(DISTINCT d.document_id) AS document_count
        FROM sources AS s
        JOIN documents AS d ON s.source_id = d.source_id
        WHERE d.publication_year >= :first_year
        GROUP BY s.source_id, s.source_title
        ORDER BY document_count DESC, s.source_title
        LIMIT 20
        """
    ),
    engine,
    params={"first_year": 2020},
)
recent_sources

In [ ]:
# La respuesta queda disponible para otra aplicación sin repetir la conexión ni la consulta.

recent_sources.to_csv(SUBMISSION_DIR / "recent_sources.csv", index=False)
questions = [{"pregunta": "¿Qué fuentes concentran documentos de proptech publicados desde 2020?", "archivo_respuesta": "recent_sources.csv"}]
(SUBMISSION_DIR / "questions.json").write_text(json.dumps(questions, ensure_ascii=False, indent=2), encoding="utf-8")
engine.dispose()